# Stardox Email Scraper

Takes a list of GitHub usernames and scrapes their public email addresses using a headless browser.

**How it works:**
1. Spins up headless Chromium via Playwright (downloads its own browser — no system Chrome needed)
2. Visits each user's GitHub profile
3. Looks for email in the profile sidebar (JS-rendered)
4. If no email on profile, checks their commit history (.patch files)
5. Outputs username:email pairs as a downloadable CSV

In [13]:
# Install Playwright + its own bundled Chromium (does NOT use system Chrome)
!pip install -q playwright nest_asyncio pandas tqdm
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [ ]:
import re
import asyncio
import nest_asyncio
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright

nest_asyncio.apply()

EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
IGNORE_PATTERNS = ['noreply', 'users.noreply.github.com', 'github.com', 'githubusercontent']


def is_valid_email(email):
    if not email:
        return False
    email_lower = email.lower()
    for pattern in IGNORE_PATTERNS:
        if pattern in email_lower:
            return False
    return True


async def start_browser(github_cookie=None):
    """Launch headless Chromium and return (pw, browser, context)."""
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)

    if github_cookie:
        context = await browser.new_context()
        await context.add_cookies([{
            'name': 'user_session',
            'value': github_cookie,
            'domain': '.github.com',
            'path': '/',
            'secure': True,
        }])
        print('Browser started with GitHub session!')
    else:
        context = await browser.new_context()
        print('Browser started (anonymous — profile emails will be hidden)')

    return pw, browser, context


async def stop_browser(pw, browser):
    """Clean up browser and playwright."""
    try:
        await browser.close()
    except Exception:
        pass
    try:
        await pw.stop()
    except Exception:
        pass


async def scrape_email_from_profile(page, username):
    """Visit GitHub profile and extract email from page text."""
    try:
        await page.goto(f'https://github.com/{username}', wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(2000)

        body_text = await page.inner_text('body')

        emails = EMAIL_RE.findall(body_text)
        for email in emails:
            if is_valid_email(email):
                return email

        source = await page.content()
        mailto_matches = re.findall(r'mailto:([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})', source)
        for email in mailto_matches:
            if is_valid_email(email):
                return email

    except Exception:
        pass

    return None


async def scrape_email_from_commits(page, username):
    """Get email from user's commit .patch files."""
    try:
        await page.goto(f'https://github.com/{username}?tab=repositories&type=source',
                         wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(1000)

        repo_elements = await page.query_selector_all('a[itemprop="name codeRepository"]')
        repo_names = []
        for el in repo_elements[:3]:
            name = await el.inner_text()
            repo_names.append(name.strip())

        if not repo_names:
            return None

        for repo_name in repo_names:
            try:
                await page.goto(
                    f'https://github.com/{username}/{repo_name}/commits?author={username}',
                    wait_until='networkidle', timeout=20000)
                await page.wait_for_timeout(1000)

                commit_links = await page.query_selector_all(
                    f'a[href*="/{username}/{repo_name}/commit/"]')

                for commit_link in commit_links[:5]:
                    href = await commit_link.get_attribute('href')
                    if not href or '/commit/' not in href:
                        continue

                    label = (await commit_link.get_attribute('aria-label')) or ''
                    text = (await commit_link.inner_text()) or ''
                    if 'merge' in label.lower() or 'merge' in text.lower():
                        continue

                    if href.startswith('/'):
                        href = 'https://github.com' + href

                    await page.goto(href + '.patch', timeout=15000)
                    await page.wait_for_timeout(1000)

                    page_text = await page.content()

                    from_match = re.search(r'From:.*?<([^>]+@[^>]+)>', page_text)
                    if from_match:
                        email = from_match.group(1)
                        if is_valid_email(email):
                            return email

                    emails = EMAIL_RE.findall(page_text[:3000])
                    for email in emails:
                        if is_valid_email(email):
                            return email

            except Exception:
                continue

    except Exception:
        pass

    return None


async def scrape_email(page, username):
    """Try profile first, then commits."""
    email = await scrape_email_from_profile(page, username)
    if email:
        return email
    return await scrape_email_from_commits(page, username)


async def worker(worker_id, context, queue, results, pbar):
    """Worker that opens its own page and pulls usernames from the queue."""
    page = await context.new_page()
    try:
        while True:
            try:
                username = queue.get_nowait()
            except asyncio.QueueEmpty:
                break

            email = await scrape_email(page, username)
            results.append({'username': username, 'email': email})

            if email:
                print(f'  [W{worker_id}] ✓ {username} -> {email}')
            else:
                print(f'  [W{worker_id}] ✗ {username} -> not found')

            pbar.update(1)
            await page.wait_for_timeout(1000)  # be polite
    finally:
        await page.close()


print('Functions loaded. Ready to scrape.')

In [ ]:
# ===========================================
# GITHUB SESSION COOKIE (required to see profile emails)
#
# How to get it:
# 1. Log into github.com in your browser
# 2. Open DevTools (F12) -> Application -> Cookies -> github.com
# 3. Find the "user_session" cookie and copy its value
# ===========================================

GITHUB_COOKIE = ""  # paste your user_session cookie value here

# ===========================================
# PASTE YOUR USERNAMES BELOW (one per line)
# ===========================================

usernames_input = """
lcbusby
decapostos
GrsmvD
DellDi
hirokikusakabe
Pankaj-sre
taoguan
sathya-ml
TrickRiggin
dhiraj-satish-bhosale
whenxtra22-ship-it
YounusBayeta
889-dj
Mahad444
dotslash21
cpedia
thelinuxer
ha-mark
Traderfuz
Tarun-Talreja
yerang1
Hirohqv
P-yoganjaneyulu
jovarela
nullpointers
gonsis10
brianrabil
dasdebanna
leonardoaires2207-cell
Desperado-Jia
YeLuo45
Amodellis
olcs
openclawq
icodebuster
adilfurkanekici
splshdown
NG90266
superadvisor007
purusho-390
IA-35
0xbageltoes
kukei777
zmctinas
AaronLiu-713
bonvoyage03
kvelkov
nnons
pintelligence9000
Ray-1214
SamNotFound697
Grigsuv
byg2025
dingxiaowei
472071378
luke886655
abhiram-vad
NeuralBlitz
ShiyuChai
patrickchen2305-design
bajgai
brandondees
skmcgov
P-shuaiwang
lwbnb233
JustToFaith
ABCbum
sruckh
jac697
fabriziorodrigues
Vilhelm-theone
beshr11
DalsZCR
KoutosGR
JAWz3D
jnews456
SankalpKat
AshishGapat
alexjacquet
skmundra
vergleb
Pure-Darkness
CasualJon
nekhiliomarcherif
coachpo
sandeeppati211
umtlondon
GYCai-tech
razor54
MassBklf
elnicomaldonado
paraself
bastiaan13345
STLtianshen
Sayem99999
GoldenPoop
kartikkabadi
herotwo7
coldlikedecember
takatamakata
amalano
jiopl1234
iamsk
ignatiousagile
nvtrr0
prenansantana
aguzik
manhnt92
greggmojica
ZenFreedomLove
NathanNg168
Bazaid
Klaas33
lxcsjk
Plague1313
Leon555
yiniesta
fi21-bot
Qiyunqun
rabhub
C5-jpg
abobakeralsaraf
rvalen1123
alexdotpink
tigdf
allababbot
tylerwang26
Deathbox5748
awezio
tanner-chiang
Alexander-J-Quant
hafsi123
archyqx
shakaluu
kaihuaybj
gunvir103
itscooleric
04cb
simonfishgit
billtill
vinnyduke
overtron
ffidan61
vide-ops-ai
P0keo
CKEN-STAR
satrijandi
bpaulus-source
ukaia
bl1nkr1
agkediri
jordipala
FatihBasim
NAME0x0
k13-za
XiaoxiamiDongfang
Semiramissis
LuxDevNet
atrim11
serkin
emirH00
HollowolloH
kasjens
morristech
darkkean
shahzada
BlackStrayCat
bgitit
omrabt
zaengerlein
lollo9427
sebastianliu
joeysim
viniciussilvaramos
ZephyrDDD
kurigashvili-godaddy
nevrget
Mohit21GoJs
dandevaud
CrashOzX
bmj-sys
szz65
mayanmy
itsnavee
Jamestone
gosrivats
zzhsimon427
fernando1987
debarghya12
highLuo
nextify2025
Serali-Nerri
Agala82
suisuss
zachgenius
realkoushikkc-ship-it
EVESIS111
JWhiteUX
Josiane-tech
auschi-shieldai
yosiwizman
PittXX
itsmastabruce
fernandoadrianfonseca
whitebaby
jay01010
Rogan-X
IUgly
AkiMonike
Wyntuition
getumen
wuming333666
hide-mel
wood-run
snackmonn
handsomeclear
Elnino69cubite
ArsenBarsen501
mateuszlewko
atthakorn
DigitailSaint54
kuravista
celandeniz
skacper
jverdeyen
vishbin
Geowahaha
cmayo93291
Eulersnmbr
nesacodes
dtjgp
3bm
dovi20
ArrichM
Karuisawa-Mrs
abdul
harvard-degen
jeffschuler
starmido
Psnt
stellarrover
awsm-vanessa
uh-joan
Wei-Xia
astronomychm
shadowofgost
kelvintanwm
kalyanv1a
yoshinaga2015
samizwei
AGLcaicai
wsl66
shoumu
immortal-oe
kjk81
thoomi2009
Kurosame777
wanghanzhen
bzfq21
ankur-continue
thesovereignflamecompany
surya-lk
MihirModi1421
artificialtheory
dwi2
iFuon
imusmanmalik
rehypothecation
rachelggao-commits
Steven-cpp
oldcrabafc
djdefi
HDZTony
Maverobot
lizhexun
jefflaporte
naokicc
jiteshpabla
conancheng
DanTerepka
musaqasenfilho-hue
guswns0105-wq
ravitejalanka
SunG206
quokka-commit
KHao123
mpalomino
aigcenc
Fram-Jam
nasserml
Szymok
nofucksrgiven
Si42
GH001-thr
xiaolongguo
tribixbite
amazing-gao
haolei
alenkovacevic
luengnat
HoodRatThings
austinembry5-del
amer2040
Funkmnk
Kpowered
CORPO-RATE
MLTurtle
sondre-hk
zacharycvivian
metatools2023
az0307
SZoloth
samkovacs
amat27
neocody
danieljrgde
XBXyftx
jj1985
deeprightai
rushi404
Genesis-50
doublebag
phungdo
perusingperuse
kamarmack
leslieyeo
jtoledom1
waxjoy
bot-unit
greatfreedom
underdog68
Stefwewo
edifierx666
dxmanoo
regional-specter
ertingxihu
Newverse-Wiki
CourtimusPrime
loganhorton
kakagogo
Ricksy1024
D3kion
arsialabs
Daniel-Humberto
piklen
romainvienne
wb3rg
edwardfernandes
raphael-cohen
LeeJSwit-HCN
titanism
1991-1005
OsayBrodie
kkkkkkkkk16
hareluya
ahmedmusharaf31
JerrYao-Cap
ddgszc
gaoyushuang
sureshreddy197
shuhei909
akoweicollinxx
anant-rustagi
Wenle-xoxo
nerkomay
Panrui423
jeffe
5angXR
abunimeh
meetrust
shing100
lechatlisse
saltymaccas
Francesco502
AaronLee184
WLCS123
jenul-ferdinand
shota
pakholeung37
romancelll
fendogo
MarinFire
PrimeMeridianETH
ryanxia2016
izetaiota
WolfricWang
niravGanatra
borhan-kh
AsifMushtaq
justitdage
BoomYoung
tuanductran
fakuzao
krishnanaredla
xiaodongzai
XiaoTianJianJun
baoguanwen
marcoscannabrava
hugefisco94
cyx3212
terry-an-investor
xup3rr
teplineL
nkeat12
taatiq882
ervitis
Syedsaulat
thurft
Southern-Upstairs-73
kiooo16
ymcool2000
duncan-G
kvsrh
techflux
astrowq
antunaesclusa
naga12dev
vinsgte
FredK8
JBrandonS
wildwulfie427
nsina
KnowledgeHog008
Sentraic
mauriciotrevinosa-cell
iroko537
jrladwig
chafiaya920-del
Max965
y0x
patilanupam
khanhn2000
pradeesh-kumar-17
Yaso2Go
gogonuk
sidhunt
Johnymachettes
saim-shipu
graysonhyc
MA22DE
OscarPowell
kmajo
DanSmaR
K1andK88
gogoqaz
dandan2611
cdchamness
quanwenxing
ModiCoding
lzchen1998
b1nhm1nh
rtpacks
markblz
Horr-Joel
exceltechai
andrewthomastaylor
ZerionL
jiamingzhang36
TheMapleseed
QilinGu
leminhhai
Ronit33
allenlai816
NonPCTH
Dorrian
borbert
zldoty
andreiaptarifa
vhild
julie-berlin
jvasquez9729
Zam1r
thetkoseek
thalesvalente
mts2350
hipa211a
leo-grnd
immuneinsanity
davidbessenyei
yfcool
diegodscamara
austinle99
joycemeet19981020
f2dmarques
FrederickPi1969
Tom-Gold
britneywwc
L-HIT-D
xhwhis
itamarEarly
kusnezoff-alexander
leo-bjorn
derbronko
dmitri-lerko
fokusferit
drupalshift
tlepk
santiagoahc
einarra
consigli3r3
greydelta
meijianwei8
stefanocalabrese
mantalope1995
renner3103
hurradieweltgehtunter
MRDavidYen
addu1990
x-tong
Riim
anandamantha
jwnys
riturajFi
singlepane-io
vams2krish
ylh990835774
krankos
hcc5
iammatthi
orferch
RtKelleher
UMMAN2005
Yimsang-Yu
StrayDragon
pratikgujral
EvolveAI-Adapt
kevinngr
whongli110-a11y
alvachen2024
gtwww
CloneXpert
Symmaque
chief-fei
Adarsh0901
hashvibe007
wanli15nian
3ba2ii
yaminsadik
sandeep-devarapalli
ortisan
ktpttd
jofongang
subzeroflame
ramikhafagi96
PrecipiceBlades
sickate
Schumi543
nboliv
zayminmaw
mihirpatel1112
vhbsouza
MitrKay
jdbence
ZiyanaliSaiyed
datwonamitchell
kapurohi
rushabhkhandhar
liyupeng111
msavdert
faint45
LBB2005
regnna
ishtiaque05
Frostburn2332
myy1966
xiangao1
cruzantony
RoseLV
0xByteBard404
MorseWayne
Knull99-hub
badrelmazaz
LinJTF
HeBai97
1ShaikhAsif
technofreak
Rana-X
yaotongsb
AlisterBaroi
Evads-Git
juancavanagh
peng925
Terencesun
ct-ferozakbar
johnboes
haiCheng-76
vgandhi13
isabelmoore
pezy
ctyeh
jericho348
danielf-rodriguez
dciconi
JakeKang
gitforpushpak
anas-aqeel
hsynatalay
smokyngt
Arracheur2Bonnet
aosherov-droid
ayvemake
vikassah
Rhushya
ejayny27
nickvo08
amitpoorab
tyalakat
NicoMancinelli
CreativeTab
meoral99
klysman08
a-mashhoor
nsaracino02
juandelich
chrisa182
jpmti2016
Juancorreav
labarilerodrigo
PossumXI
FuncGuy
VikalpRajKisku
imtiaz84-cmyk
golfoo
pshoukry
objectorientedperson
TazRT
mbushey1
IanMariacaC
ebaville
gokay
Dinesh-Sheelam
jonmarrs
saschad
winstonpgao
bunimarko
KriZa96
thebenignhacker
jose-caban
eduardocornelsen
chrisbergeron
coconut-junior
varbinaryequals
anthonylinartemis
comickhan
amejri
shkao
whoismarios
vercotte
mahm81
V3n0m85
rsiedlar
ramuroN
flimble
nayasavi
MarioMaldonadoH
eunkomeme
achraf-gasmi
a-aznar
EDLuke
drsrk4u
dmitriyzhirma
KillrBee
marcostx
AliKrtgl
sharka2006
mdsakalu
Raghavan-27-5
NVRGNNAWIN
polemos74
spector-in-london
CesarPetrescu
lhpl172005
Svenzhan
themagrus
qiwi-qiwi
Seraph1911
gitdekado
Amaimaya
nuoomnoy02
ErayErman
PavanMutyla
rsohlot
filipkovachevski-source
bsormagec
gaktug
trungitk
donovinsims
kgcrom
diaswrd
Anionix
Aznatkoiny
Panjkrc
texanraj
masknugget
gledun
caoxuecheng
GongyanL
isengartz
119533564
upa8
LiCHT-77
wzk-0826
emircanagac
sedataziz
TheGreatestGgoat
jihwan38
zjw57
MrR0990
caodchuong312
Oleks1y
chinazc688
RobinLi2000
ham0806
sfinning
changqingniubi
busingepius
lucemia
iforlife
Elwinc2799
Franky1
xuyi
zhangtesla
omar-nuhman
muhammedfurkansahin
lafanadaer
yangdaowan
gilanggp
timvdhoorn
JKL999
luiscuellar31
www-xu
dsyoon-crypto
guanyunhao
472027909
Sheng723
Sth-AI
orangewings97
uniinu1
zoan37
fahrigedik
interrogaviter-cmd
draconian-delight
cisjzs
wheesys
zhangganhao
ehiluz
quwang123
YourNPCer
EasyCommanderZ
btasseh-beep
97vack
saidzia
DanielMax937
baipoonnn
YuChenu
jurgenkurti26
wangyanhui2023-max
PhilosLiOfficial
hoanglongle88
sambonner
saransh07-a
Kaweees
03Trash-Panda31
gjija
anderson
ozkay34
FMENGD
matissime
neuroloom
chima222
echelonxelite
kidd255
sceboucher
Khwayld
Pedrooro
fever365
yutasth
adilmarruiz
originalconsultoria
parsiyel
zhangliwen2014
shogochiai
stellarthemes
MaxOnceProject
ariburaco
replikduplik
herrontrading
anagparedes
Pioos
cberktavsan
youcefaddou
slachiewicz
pavstev
sbehbxl
manifestcowboy
ebicoglu
fffedor
mrkfks
Monolithcreative
Buschbraue
mjumhah
balazsbrczk29-rgb
ferdiunal
richardoberkofler
gansxx
shehral
jryjry-ui
lukedry
bllkrkmz
HacktopusX
gurkangul
Safa675
dominik1001
serkanyasr
Jinaprasad
yellohman
Harmish-Data
deltacompetitions
AlbertNjobo
zsnmwy
theronic
leondev82
david43
gupta-shashank
Hakaczu
masa206rs
ducky-labs
johnzzj
thedawn112
worldoftoddl
masclown
huseKivrak
m0x1c4c4tu4
thalesdelmiro
faxxxxx
BugOfTime
na-xn
0xAlexandeeer
jun9lee
N7Ren
gestas666
noguchi-im
datadriven-saurabh
tomodern
HarKro753
msariakcali
davea38
feliperabeloep
JulianAnmoon
napnel
dmitryn
nivance
alivemike
YungBenn
d3v3l0
k32ru
EVTKR
kaznak
Hayato-T-08
aamorozo
c-yokoyama
puglylife
afeicool
harishpiyer
l15c
bodhi-crypo
manboubird
6uclz1
dadachi
ArchieHicklin
ashigirl96
tunotuki
leoyb
brsmrc
GalBerezansky
prophet555
mkusaka
hiroingk
Sunalamye
hellofriendyoumad
ardhayosef
Burakylcnygt
oruga159
sugihara1997
leonardomso
CharlesTango
Designer636
hbudin
yu-iskw
xt0x
Powersup8
leadsyncd
minh25
mavmateo
noriHanda
maxto
uooooo
terudoru
Claudoi
sumitkumarm
Gudong070
mashirohaku
DiyarZhunussov
u1aryz
TomokiIchi
vincentweisser
TheWallStreeter
SnoW248
Hiroto555
TakaoWing
oggiultra-tech
userhasaccess
kyvu
zhutopian
adarshbiradar07
ArbezRM
huming1420
MrXploisLite
9amcoder
giobirkelund
SiixQuant
leanheon
franklinmdev
xpiggyy
10chim
harisfarrasi
AgarwalAarush
sanjigrillshoot7-gif
mragusa
appff
ewyluda
taichi0001
ani-code
vinhisme
mAengo31
lordyeagh
Mossayl
bharat2288
samon6bl
J0hnG4lt
HaithamMaya
spyrux
sushikev
leethomas
RichardWang812
VineedKaladharan
yhzami
lpandeloc-ctrl
OplopUser
pranay1409
lrkkr
r1ndl1re
hq-4
glassBead-tc
aneeshka01
boydstor
sachins-scribd
howardshim
mavrekc
alexxanderdiaz
chminsc
Ltbltbltbltb
kobi-ca
Knowledgesynthesis
MrZoder
TheCrypt0An0n
pzerodev
billyndroid
MarselProg2
chinalcm001
grizzly-claw
d-sumo-boy
Velestial
racasado
djbijo
TMTrevisan
hchoi41
imjorgemor
toonsikosek
avivsinai
LongZ-A
docusiva
nginz
pachodomi
saberzaid
goodlookingcat
gtdrag
maxcodes
ytinoooon
chethanuk
AmaleshV
MyButtermilk
VyasKS
AtomicaInvest
datboi2001
Bencodes
Mukai313
yilmaz-burak
seetinc10
zaakirio
toga0731
leonardo-vanschaik
PS-Graham
jongan69
SteveActual
jshn9515
oreganoflakesgit
fclesio
chenrui333
icsrick
wanghulouxia
atita2026
pmryan
hyx4992
minhluudinh
100868871
esaiz
ellisd4488-del
alindsilva
21iridescent
beamsley
Paysonpang
srvo
data4sci
orkuny1
maazullah96
wplf
adriannacrai413-png
zxc74105
luca8am
poferraz
SHINJOAH
RauPtr
horizon-old-daddy
thinkriver
jibzus
geek-shawn
kcorn23
omgnuts
stvlynn
kanekanefy
babybank25
chengjunjian
ngyngcphu
josephfkrause
evelynmitchell
alphabravo1217
firelife
StevenRidder
sbhavani
tjweir
akhmedov777
NAlle33
a-proenca
pdbennett
the-real-adammork
tinyspacelobster
JamesHuang22
mattdrinkwaterb
binlialfie
zaherarman
AgustinAlbonico
Canhha
smallball300
ShawnRoller
aaronvan25
zhangzhg0508
ashishthukral
zzzh1hao01
guinness74
CORCTON
naghdy
cherlin
jimmylin0979
KyleCurtisBiz
halojunkie
danielpang
joshuaramkissoon
0xD7ba952CE8A0976e8d9852b7649bf01c30146
SrSh-notme
ankuragupta
aspatil2
Oliclee
Magabtc2025
johntheyoung
mohsalsaleem
chengdujin
Hao-06
unlimitedtsar
eloiferrer
marmot63
orest22
tynes
gggggnaw
jagatheesh31
ktrawick
Jo1ceLi
hossein761
alexyu0814
faizanke
aggiee
Sneaker001333
muaadh0077
ateesdalejr
gaoerrong
sureindia-in
de-caff
charlyk-code
liq07lzucn
huangdi614
rimenes
sofistycstudio-ship-it
JWTseng
d234mkikodo
gq23401
arghosh
Virtuallaborer
x1automent
startagain2016
philipandersson
packalyst
ankitsablok89
ZX-VSC2025
ctyytc
YinFY90
levinas76
one1d
User290663
rodolfoasantos
metefire
wesleysmyth
vanisett1
abogushov
zico-source
turnDeep
m3hr4d
hoksilato
jorgeavilacardenosa
qiqi1qiqi
leotsaiCode
diegomillan
InvestSaisPas
luishenriquejm
ZacBi
pikinkz
rule620
sztwt
dbarden
leishao
neetsethia
ygqwan
zhouhao
duhow
MateusAndrade
vbalazs
albertparis
inrust
leeeandroo
ravitej-pudi
ehkluo
thedileepkumar-dk
floris-xlx
kerpopule
joshiujjwal
AdamBRam
Mhod-gad
kennethh72
TM0088
jyjoo94
yifeizhang-hku
Acatsama0871
junejueki-dev
kenchangx
pkaysantana
henry11996
SathishN
Erickrus
sportsandfragrance
naiFeiTian007
leastwanted
FredLIU2000
ifandelse
tycoonBB
baonguyen1904
rajesh-chawla
dchong1
ElationNest
jdbice
renzit
CorySalmon
xXt0rm
aZeBy
MiluoGreatWall
chadleong
privatelogic
leonlowitzki
snowcodeer
sbbddz
psotoulloa
ryanstorandt
darunfafishkiller
ashknl
caifengsteven
HugoGDO
3plusalpha
MoneyDotCom
cezarlica
nh2seven
zidny-z
alexbaldwin
lite
bnpinel
berkode
YLeemm
niaccky
arnaud-zg
victor504717-afk
vinerya
Ceasoul
weiweiyoudianpang
Captain-Nuc
xywo
whitemachine
tkudotdev
"""

# ===========================================

usernames = [u.strip() for u in usernames_input.strip().split('\n') if u.strip()]
print(f'Loaded {len(usernames)} usernames')
if GITHUB_COOKIE:
    print('GitHub cookie provided — will see profile emails')
else:
    print('No GitHub cookie — will only get emails from commit history')

Loaded 1633 usernames
No GitHub cookie — will only get emails from commit history


In [ ]:
# Run the scraper — 5 parallel workers (5 browser tabs)
WORKERS = 5

pw, browser, context = await start_browser(github_cookie=GITHUB_COOKIE if GITHUB_COOKIE else None)
results = []

queue = asyncio.Queue()
for username in usernames:
    queue.put_nowait(username)

try:
    with tqdm(total=len(usernames), desc='Scraping emails') as pbar:
        tasks = [worker(i, context, queue, results, pbar) for i in range(WORKERS)]
        await asyncio.gather(*tasks)
finally:
    await stop_browser(pw, browser)

found_count = sum(1 for r in results if r['email'])
print(f'\nDone! Found {found_count}/{len(usernames)} emails')

In [17]:
# Results
df = pd.DataFrame(results)
print(f'Total: {len(df)}')
print(f'With email: {df["email"].notna().sum()}')
print(f'Without email: {df["email"].isna().sum()}')
print()

# Show all results
display(df)

# Save and download CSV
csv_filename = 'stargazer_emails.csv'
df.to_csv(csv_filename, index=False)

from google.colab import files
files.download(csv_filename)
print(f'\nDownloading {csv_filename}...')

Total: 56
With email: 30
Without email: 26



,username,email
0,3rdAI-admin,None
1,ArkayaVenture,admin@arkayaventure.co.uk
2,Kaairofelipe,None
3,cpiprint,tcarson@cpiprint.com
4,cpiprint,tcarson@cpiprint.com
5,cpiprint,tcarson@cpiprint.com
6,cpiprint,tcarson@cpiprint.com
7,shreyasgm,shreyas.gm61@gmail.com
8,asfakahamedc,None
9,jsairdrop1,None


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>